# McLean multi-cluster storm product view

This is the product version of the May 27, 2024 multi-storm case. It combines the real three-radar McLean mosaic with every active KLWX cluster track, then reports the combined chance that **any** tracked reflectivity cluster covers the McLean office.

The output is a radar-cluster coverage baseline, not a validated probability of severe weather or damage. The 30-minute and 1-hour horizons are the useful baseline-nowcast windows; 4/8-hour values are exploratory constant-velocity projections.

In [ ]:
%pip install -e "../.[nexrad]"

In [ ]:
from datetime import timedelta
from pathlib import Path
import sys
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import matplotlib.pyplot as plt
import numpy as np

repo = Path.cwd().parent
sys.path.insert(0, str(repo / 'src'))
from storm_motion.impact import equivalent_cluster_radius_m, local_xy_from_lat_lon, point_impact_probability
from storm_motion.mosaic import max_reflectivity_mosaic
from storm_motion.nexrad import load_lowest_reflectivity_sweep
from storm_motion.tracking import MultiClusterTracker, RadarFrame, cluster_cells, detect_cells

bucket = 'unidata-nexrad-level2'
klwx_keys = [
    '2024/05/27/KLWX/KLWX20240527_221509_V06',
    '2024/05/27/KLWX/KLWX20240527_222142_V06',
    '2024/05/27/KLWX/KLWX20240527_222826_V06',
    '2024/05/27/KLWX/KLWX20240527_223511_V06',
    '2024/05/27/KLWX/KLWX20240527_224155_V06',
    '2024/05/27/KLWX/KLWX20240527_224840_V06',
]
data_dir = repo / 'data' / 'klwx_20240527_multicluster'
data_dir.mkdir(parents=True, exist_ok=True)
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))
def download(key):
    path = data_dir / Path(key).name
    if not path.exists():
        s3.download_file(bucket, key, str(path))
    return path

klwx_paths = [download(key) for key in klwx_keys]
frames = [load_lowest_reflectivity_sweep(path) for path in klwx_paths]

In [ ]:
roi_km = dict(xmin=-35, xmax=125, ymin=-45, ymax=120)
def in_roi(frame):
    x_km, y_km = frame.x_m / 1_000, frame.y_m / 1_000
    inside = (x_km >= roi_km['xmin']) & (x_km <= roi_km['xmax']) & (y_km >= roi_km['ymin']) & (y_km <= roi_km['ymax'])
    return RadarFrame(frame.observed_at, np.where(inside, frame.reflectivity_dbz, np.nan), frame.x_m, frame.y_m, frame.radar_latitude_deg, frame.radar_longitude_deg)

def clusters_for(frame):
    for threshold_dbz, min_gates in [(40, 12), (35, 16), (30, 20)]:
        clusters = cluster_cells(detect_cells(frame, threshold_dbz=threshold_dbz, min_gates=min_gates), linkage_distance_m=10_000)
        if len(clusters) >= 2:
            return clusters, threshold_dbz
    return clusters, threshold_dbz

roi_frames = [in_roi(frame) for frame in frames]
final_source_frame = frames[-1]
del frames
cluster_results = [clusters_for(frame) for frame in roi_frames]
multi_tracker = MultiClusterTracker(measurement_std_m=4_000, acceleration_std_mps2=0.01, association_distance_m=30_000, max_missed_scans=1)
updates = [multi_tracker.update(frame.observed_at, clusters) for frame, (clusters, _) in zip(roi_frames, cluster_results)]
for frame, update in zip(roi_frames, updates):
    print(f'{frame.observed_at:%H:%M:%S UTC}: ' + ', '.join(f'T{track.track_id}:{track.status}' for track in update.tracks))

In [ ]:
# Fuse time-nearest scans at the final KLWX time, on the focused McLean-local grid.
office_latitude_deg, office_longitude_deg = 38.922403, -77.232125
office_x_m, office_y_m = local_xy_from_lat_lon(office_latitude_deg, office_longitude_deg, final_source_frame.radar_latitude_deg, final_source_frame.radar_longitude_deg)
mosaic_keys = [
    '2024/05/27/KDOX/KDOX20240527_224828_V06',
    '2024/05/27/KAKQ/KAKQ20240527_224647_V06',
]
mosaic_frames = [final_source_frame, *(load_lowest_reflectivity_sweep(download(key)) for key in mosaic_keys)]
for frame in mosaic_frames:
    print(f'{frame.observed_at:%H:%M:%S UTC}: {(frame.observed_at - final_source_frame.observed_at).total_seconds() / 60:+.1f} min from final KLWX scan')
local_radius_km, local_resolution_m = 25, 250
local_x_m = np.arange(-25_000, 25_000 + local_resolution_m, local_resolution_m)
local_y_m = np.arange(-25_000, 25_000 + local_resolution_m, local_resolution_m)
local_mosaic_dbz = max_reflectivity_mosaic(mosaic_frames, origin_latitude_deg=office_latitude_deg, origin_longitude_deg=office_longitude_deg, x_m=local_x_m, y_m=local_y_m)

In [ ]:
# Forecast every active track, then combine per-track coverage probabilities as P(any)=1−∏(1−p).
last_time = updates[-1].observed_at
range_step_m = np.nanmedian(np.diff(np.hypot(final_source_frame.x_m[0], final_source_frame.y_m[0])))
azimuths_rad = np.unwrap(np.arctan2(final_source_frame.x_m[:, 0], final_source_frame.y_m[:, 0]))
azimuth_step_rad = np.nanmedian(np.abs(np.diff(azimuths_rad)))
forecast_minutes = [30, 60, 240, 480]
def lead_label(minutes):
    return f'+{minutes // 60} hour' if minutes % 60 == 0 else f'+{minutes} min'
outlooks = {}
for minutes in forecast_minutes:
    tracks = multi_tracker.predict(last_time + timedelta(minutes=minutes))
    impacts = []
    for track in tracks:
        cluster = track.last_observed_cluster
        gate_area_m2 = np.hypot(cluster.centroid_x_m, cluster.centroid_y_m) * range_step_m * azimuth_step_rad
        radius_m = equivalent_cluster_radius_m(cluster, gate_area_m2)
        impact = point_impact_probability(track.estimate, office_x_m, office_y_m, radius_m, random_seed=(minutes * 100) + track.track_id)
        impacts.append((track, impact, radius_m))
    combined_probability = 1 - np.prod([1 - impact.probability for _, impact, _ in impacts]) if impacts else 0.0
    outlooks[minutes] = (impacts, combined_probability)

print('MCLEAN MULTI-CLUSTER IMPACT OUTLOOK')
for minutes, (impacts, combined_probability) in outlooks.items():
    confidence = 'baseline nowcast' if minutes <= 60 else 'exploratory constant-velocity projection'
    contributors = ', '.join(f'T{track.track_id} {impact.probability:.1%}' for track, impact, _ in impacts) or 'no active tracks'
    print(f'  {lead_label(minutes):>8}: {combined_probability:5.1%} chance any tracked cluster covers McLean ({confidence})')
    print(f'             per-track: {contributors}')

In [ ]:
# Product decision view: actual fused radar plus every cluster's +30 minute forecast.
fig, ax = plt.subplots(figsize=(10, 9))
image = ax.pcolormesh(local_x_m / 1_000, local_y_m / 1_000, local_mosaic_dbz, cmap='turbo', vmin=0, vmax=65, shading='nearest')
ax.scatter(0, 0, marker='*', s=420, c='#ff4d4d', edgecolors='white', linewidths=1.4, zorder=10, label='Booz Allen · McLean')
colors = plt.get_cmap('tab20')
for track, impact, radius_m in outlooks[30][0]:
    color = colors((track.track_id - 1) % 20)
    x_km, y_km = (track.estimate.x_m - office_x_m) / 1_000, (track.estimate.y_m - office_y_m) / 1_000
    ax.scatter(x_km, y_km, s=150, c=[color], edgecolors='white', linewidths=1.2, zorder=8)
    ax.add_patch(plt.Circle((x_km, y_km), radius_m / 1_000, fill=False, edgecolor=color, linewidth=2.4, zorder=7))
    ax.add_patch(plt.Circle((x_km, y_km), track.estimate.position_std_m / 1_000, fill=False, edgecolor='white', linewidth=2.2, linestyle='--', alpha=.9, zorder=6))
    ax.annotate(f'T{track.track_id} · {impact.probability:.0%}', (x_km, y_km), xytext=(7, 7), textcoords='offset points', color='white', weight='bold')
ax.axhline(0, color='white', alpha=.3, linewidth=.9); ax.axvline(0, color='white', alpha=.3, linewidth=.9)
ax.set(title=f'McLean: all active clusters at +30 min · combined coverage {outlooks[30][1]:.0%}', xlabel='km east (+) / west (−) of office', ylabel='km north (+) / south (−) of office', xlim=(-local_radius_km, local_radius_km), ylim=(-local_radius_km, local_radius_km), aspect='equal')
fig.colorbar(image, ax=ax, label='maximum observed reflectivity (dBZ)'); plt.show(); plt.close(fig)
print('Colored circles: one forecast per active cluster. Solid colored outline: estimated footprint. White dashed: 1σ center uncertainty. Combined percentage assumes per-track errors are independent.')

## McLean regional decision view: 50 km around the office

The map above spans ±25 km from McLean (50 × 50 km total). This broader view spans ±50 km (100 × 100 km total) and is re-fused directly from the three original radar scans, so it keeps the source detail instead of enlarging the smaller map.

In [ ]:
regional_radius_km = 50
regional_x_m = np.arange(-regional_radius_km * 1_000, (regional_radius_km * 1_000) + local_resolution_m, local_resolution_m)
regional_y_m = np.arange(-regional_radius_km * 1_000, (regional_radius_km * 1_000) + local_resolution_m, local_resolution_m)
regional_mosaic_dbz = max_reflectivity_mosaic(
    mosaic_frames, origin_latitude_deg=office_latitude_deg, origin_longitude_deg=office_longitude_deg,
    x_m=regional_x_m, y_m=regional_y_m,
)

fig, ax = plt.subplots(figsize=(10, 9))
image = ax.pcolormesh(regional_x_m / 1_000, regional_y_m / 1_000, regional_mosaic_dbz, cmap='turbo', vmin=0, vmax=65, shading='nearest')
ax.scatter(0, 0, marker='*', s=420, c='#ff4d4d', edgecolors='white', linewidths=1.4, zorder=10, label='Booz Allen · McLean')
for track, impact, radius_m in outlooks[30][0]:
    color = colors((track.track_id - 1) % 20)
    x_km, y_km = (track.estimate.x_m - office_x_m) / 1_000, (track.estimate.y_m - office_y_m) / 1_000
    ax.scatter(x_km, y_km, s=150, c=[color], edgecolors='white', linewidths=1.2, zorder=8)
    ax.add_patch(plt.Circle((x_km, y_km), radius_m / 1_000, fill=False, edgecolor=color, linewidth=2.4, zorder=7))
    ax.add_patch(plt.Circle((x_km, y_km), track.estimate.position_std_m / 1_000, fill=False, edgecolor='white', linewidth=2.2, linestyle='--', alpha=.9, zorder=6))
    ax.annotate(f'T{track.track_id} · {impact.probability:.0%}', (x_km, y_km), xytext=(7, 7), textcoords='offset points', color='white', weight='bold')
ax.axhline(0, color='white', alpha=.3, linewidth=.9); ax.axvline(0, color='white', alpha=.3, linewidth=.9)
ax.set(title=f'McLean: all active clusters at +30 min · ±{regional_radius_km} km view', xlabel='km east (+) / west (−) of office', ylabel='km north (+) / south (−) of office', xlim=(-regional_radius_km, regional_radius_km), ylim=(-regional_radius_km, regional_radius_km), aspect='equal')
fig.colorbar(image, ax=ax, label='maximum observed reflectivity (dBZ)'); plt.show(); plt.close(fig)
print('This 100 × 100 km view is re-fused at 250 m from KLWX, KDOX, and KAKQ—not a zoomed version of the smaller view.')